# 03 Feature Engineering

This notebook focuses on the feature engineering pipeline for the Sentinel Sepsis-Associated AKI (SA-AKI) Early Warning System.

## Strategy
- **Missingness Indicators**: Missing data in EHR is highly informative. We explicitly create missingness flags before imputation.
- **Rolling Windows**: To capture trends in vital signs and lab results, we compute rolling aggregates (mean, max, min, std) over **6h and 12h** windows.
- **Creatinine Velocity**: The rate of change in serum creatinine is a critical physiological marker for acute kidney injury.
- **Composite Features**: Combining related measurements (e.g., BUN-to-creatinine ratio, anion-gap proxy, MAP × creatinine-velocity interaction).

In [2]:
# Colab setup — mount Drive and cd into the project so ../data/... paths resolve.
# Safe anywhere: does nothing when not running on Colab.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    import os, sys
    PROJ = '/content/drive/MyDrive/sentinel-poc/project_sentinel'
    os.chdir(os.path.join(PROJ, 'notebooks'))
    sys.path.insert(0, PROJ)
except ModuleNotFoundError:
    pass  # not on Colab — assume already running from notebooks/
import os
print('cwd =', os.getcwd())

Mounted at /content/drive
cwd = /content/drive/MyDrive/sentinel-poc/project_sentinel/notebooks


In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys, os
sys.path.append(os.path.abspath('..'))

from src import features

interim_dir = Path('../data/interim')

print("Loading defined cohort...")
cohort_df = pd.read_parquet(interim_dir / 'cohort_defined.parquet')
print(f"Loaded {cohort_df['patient_id'].nunique()} stays, {len(cohort_df):,} rows.")

print("Loading baseline creatinine...")
baseline_cr = pd.read_parquet(interim_dir / 'baseline_creatinine.parquet')
print(f"Loaded {len(baseline_cr)} baselines.")

Loading defined cohort...
Loaded 92436 stays, 8,242,612 rows.
Loading baseline creatinine...
Loaded 92561 baselines.


In [4]:
# Subsample stays to keep memory bounded. 30k >> the 10-20k "favorable" sizing tier,
# so model quality is essentially unaffected; the full ~92k stays OOM feature engineering
# even on a High-RAM (51 GB) runtime. Seeded for reproducibility. Set N_STAYS = None to
# keep every stay (only viable with far more RAM or a batched pipeline).
N_STAYS = 30000
if N_STAYS is not None and cohort_df['patient_id'].nunique() > N_STAYS:
    keep = set(np.random.default_rng(42).choice(
        cohort_df['patient_id'].unique(), size=N_STAYS, replace=False))
    cohort_df = cohort_df[cohort_df['patient_id'].isin(keep)].copy()
    baseline_cr = baseline_cr[baseline_cr['patient_id'].isin(keep)].copy()
    print(f"Subsampled to {cohort_df['patient_id'].nunique()} stays, {len(cohort_df):,} rows.")
else:
    print(f"Using all {cohort_df['patient_id'].nunique()} stays.")

Subsampled to 30000 stays, 2,707,534 rows.


In [5]:
print("Adding missingness indicators...")
df_with_missingness = features.add_missingness_indicators(cohort_df)
print(f"Dataset shape after missingness indicators: {df_with_missingness.shape}")

Adding missingness indicators...
Dataset shape after missingness indicators: (2707534, 68)


In [6]:
print("Adding rolling features...")
df_rolling = features.add_rolling_features(df_with_missingness)
print(f"Dataset shape after rolling features: {df_rolling.shape}")

print("Imputing missing values...")
df_imputed = features.impute_values(df_rolling)
print(f"Dataset shape after imputation: {df_imputed.shape}")

Adding rolling features...
Dataset shape after rolling features: (2707534, 196)
Imputing missing values...
Dataset shape after imputation: (2707534, 196)


## Creatinine Velocity

Creatinine velocity is perhaps the most important feature for predicting SA-AKI. It captures the **rate of change** of serum creatinine over a recent **12-hour** window.

A rapid rise in creatinine is a stronger predictor of imminent kidney injury than the absolute level of creatinine alone, as it directly reflects acute loss of glomerular filtration.

In [7]:
print("Calculating creatinine velocity...")
df_cr_velocity = features.add_creatinine_velocity(df_imputed, baseline_cr)
print(f"Dataset shape after creatinine velocity: {df_cr_velocity.shape}")

Calculating creatinine velocity...
Dataset shape after creatinine velocity: (2707534, 200)


In [8]:
print("Adding composite features...")
df_composites = features.add_composite_features(df_cr_velocity)
print(f"Dataset shape after composite features: {df_composites.shape}")

Adding composite features...
Dataset shape after composite features: (2707534, 206)


In [9]:
print("Running the full feature-engineering pipeline...")
# Cells 3–7 above illustrate each step; engineer_features runs them in the
# correct order (missingness → rolling on sparse data → impute → velocity → composites).
final_features_df = features.engineer_features(cohort_df, baseline_cr)
print(f"Final dataset shape: {final_features_df.shape}")

features_path = interim_dir / 'engineered_features.parquet'
final_features_df.to_parquet(features_path, index=False)
print(f"Saved → {features_path}")

Running the full feature-engineering pipeline...
Final dataset shape: (2707534, 211)
Saved → ../data/interim/engineered_features.parquet


In [10]:
print("Creating feature manifest...")
manifest = features.create_feature_manifest(
    final_features_df, interim_dir / 'feature_manifest.csv'
)
print(f"Generated manifest with {len(manifest)} features → {interim_dir / 'feature_manifest.csv'}")
display(manifest.head(15))

Creating feature manifest...
Generated manifest with 205 features → ../data/interim/feature_manifest.csv


,name,category,pct_missing,mean,std,min,max
0,DBP,vital,4.00,64.9460,254.1965,-40.0000,1.051250e+05
1,HR,vital,1.22,85.5618,149.1131,-241395.0000,1.171000e+04
2,MAP,vital,1.72,93.9961,10909.6059,-60.0000,8.888890e+06
3,O2Sat,vital,1.68,100.1142,5501.0570,-951234.0000,8.900000e+06
4,Resp,vital,1.72,20.4812,481.3991,0.0000,7.851987e+05
5,SBP,vital,3.99,120.3787,100.5590,0.0000,1.411460e+05
6,Temp,vital,8.04,36.9901,9.5903,-17.7778,5.485000e+03
7,AST,lab,84.14,270.6690,1126.5789,0.0000,3.217000e+04
8,Alkalinephos,lab,84.43,151.7246,172.4374,7.0000,4.336000e+03
9,BUN,lab,45.83,32.2881,26.2642,1.0000,2.910000e+02


## Next Step

Engineered features are saved to `data/interim/engineered_features.parquet` (raw clinical columns are retained alongside the derived features, so labeling can still read `Creatinine`, `urine_rate`, etc.).

The chronological **train/val/test split happens in `04_label_engineering.ipynb`** — *after* the KDIGO labels are attached — so each split carries features **and** labels together with no leakage.

In [11]:
!git config --global user.email "daemmyoff1cial@gmail.com"
!git config --global user.name "Sng43"

In [12]:
%cd /content/drive/MyDrive/sentinel-poc
!git add .
!git commit -m "wip: colab feature en.. "
!git push

/content/drive/MyDrive/sentinel-poc
[main a247336] wip: colab feature en.. progress before history resync
 2 files changed, 2 insertions(+), 2 deletions(-)
 rewrite project_sentinel/notebooks/03_feature_engineering.ipynb (95%)
Enumerating objects: 11, done.
Counting objects: 100% (11/11), done.
Delta compression using up to 8 threads
Compressing objects: 100% (6/6), done.
Writing objects: 100% (6/6), 1.64 KiB | 280.00 KiB/s, done.
Total 6 (delta 5), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (5/5), completed with 5 local objects.
To https://github.com/Sng43/sentinel-poc.git
   d6c5f2f..a247336  main -> main
